These scripts are to bulk download PRISM high resolution (800m) daily climate data, unzip, and copy only .tif to a new folder.

In [18]:
import os
import requests
from datetime import datetime, timedelta
import time
import zipfile
import shutil
from tqdm import tqdm

In [19]:
#PRISM base URL for the data download as given in their website and download instructions
base_url = "https://services.nacse.org/prism/data/get/us"

#Define variables, resolution, dates, and download and output directory
#variable   = "tmax"             #Change for each variable; no need to run this if downloading multiple variables in bulk
resolution = "800m"

#Change for each year, collectively covering all fires
start_date = "2019-09-03"
end_date   = "2019-09-20"

#Defining date as required per fires date ranges each year
def date_range(start, end):
    current = start
    while current <= end:
        yield current
        current += timedelta(days=1)

start_dt = datetime.strptime(start_date, "%Y-%m-%d")
end_dt   = datetime.strptime(end_date, "%Y-%m-%d")
dates    = list(date_range(start_dt, end_dt))

#Directories
parent_dir = r"Z:\NAIP\PRISM\2019"

variables = ["tmax", "tmean", "vpdmax", "vpdmin"]      #List of variables to download, change as required

for variable in variables:
    out_dir = os.path.join(parent_dir, variable)
    os.makedirs(out_dir, exist_ok=True)

    # Loop to Download Files
    print(f"Starting PRISM {variable} download:")

    for date in tqdm(dates, desc=f"Downloading {variable}", unit="day"):
        ymd = date.strftime("%Y%m%d")

        url = f"{base_url}/{resolution}/{variable}/{ymd}"
        out_zip = os.path.join(
            out_dir, f"PRISM_{variable}_{resolution}_{ymd}.zip"
        )

        if os.path.exists(out_zip):
            continue

        r = requests.get(url, timeout=120)

        if r.status_code == 200 and len(r.content) > 1000:
            with open(out_zip, "wb") as f:
                f.write(r.content)

        # polite delay (PRISM recommendation)
        time.sleep(1)

    print(
        f"Prism download completed for {variable} from {start_date} to {end_date}"
    )

Starting PRISM tmax download:


Prism download completed for tmax from 2019-09-03 to 2019-09-20
Starting PRISM tmean download:


Prism download completed for tmean from 2019-09-03 to 2019-09-20
Starting PRISM vpdmax download:


Prism download completed for vpdmax from 2019-09-03 to 2019-09-20
Starting PRISM vpdmin download:


Prism download completed for vpdmin from 2019-09-03 to 2019-09-20


Extracting all files including .tif rasters from the zip files, copy and store in unzip sub-folder within each variable folder.

In [20]:
# Unzip Files and copy to unzip folder
for variable in os.listdir(parent_dir):
    zip_dir = os.path.join(parent_dir, variable)

    if not os.path.isdir(zip_dir):
        continue  # skip files

    unzip_dir = os.path.join(zip_dir, "unzip")
    os.makedirs(unzip_dir, exist_ok=True)

    zip_files = [f for f in os.listdir(zip_dir) if f.lower().endswith(".zip")]

    if not zip_files:
        print(f"No ZIP files found for {variable}, skipping.")
        continue

    # tqdm progress bar
    for fname in tqdm(zip_files, desc=f"Extracting PRISM {variable} files", unit="zip"):
        zip_path = os.path.join(zip_dir, fname)

        with zipfile.ZipFile(zip_path, "r") as z:
            for member in z.namelist():
                dest_path = os.path.join(unzip_dir, os.path.basename(member))
                if not os.path.exists(dest_path):
                    z.extract(member, unzip_dir)
    
    print(f"Un-zipping and copying all PRISM {variable} files is completed.\n")

Extracting PRISM tmax files: 100%|██████████| 18/18 [00:23<00:00,  1.33s/zip]


Un-zipping and copying all PRISM tmax files is completed.



Extracting PRISM tmean files: 100%|██████████| 18/18 [00:24<00:00,  1.38s/zip]


Un-zipping and copying all PRISM tmean files is completed.



Extracting PRISM vpdmax files: 100%|██████████| 18/18 [00:35<00:00,  1.96s/zip]


Un-zipping and copying all PRISM vpdmax files is completed.



Extracting PRISM vpdmin files: 100%|██████████| 18/18 [00:30<00:00,  1.72s/zip]

Un-zipping and copying all PRISM vpdmin files is completed.

